In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from datasets import load_dataset
import torch
import re
from unsloth import FastLanguageModel
from transformers import pipeline
from models import *

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
from helper_functions_MLE_fitting import load_data, num_parameters, get_bounds, nll_truncnorm, truncnorm_logpdf, fit_participant
from configurations import DOMAIN_CONFIG, OPTIM_CONFIG

In [34]:
# helper function for printing
def until_nth_occurrence(s, substring, n, num_chars_to_print=72):
    count = 0
    index = 0
    while index < len(s):
        index = s.find(substring, index)
        if index == -1:
            break
        count += 1
        if count == n:
            return s[index + len(substring) -num_chars_to_print:index + len(substring) + 10]

        index += len(substring)

    assert False, "should not happen"

In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
# TODO careful with this
used_model_id       = 1
start_simulation_id = 1

In [18]:
DOMAIN   = "Mammals"
LLM_DATA = "../Data/Preprocessed Data/narrative_data.csv"
NLL_PATH = f'../Data/Model Outputs/log_likelihood_{DOMAIN}_TESTING_ALIGNED.pth'
EXP_DATA = f'../Data/Behavioral Data/data_analysis_{DOMAIN.lower()}.csv'

if DOMAIN == "Mammals":
    from models_mammals import *
elif DOMAIN == "Food":
    from models_food import *
else:
    from models_countries import *

In [19]:
# Load data with prompt text for LLMs
dataset = load_dataset("csv", data_files=LLM_DATA)["train"]
dataset = dataset.filter(lambda x: x["domain"] == DOMAIN)
print(dataset.shape)
dataset.to_pandas().head(3)


(48, 6)


,text,domain,participant,ID,n_training,ID_items
0,Your task is to estimate the days until female...,Mammals,0,0xhc0yxm5czl,60,"76, 9, 26, 77, 70, 59, 10, 38, 43, 41, 17, 68,..."
1,Your task is to estimate the days until female...,Mammals,1,2qewr80masfh,84,"10, 26, 77, 68, 59, 17, 9, 76, 38, 43, 70, 41,..."
2,Your task is to estimate the days until female...,Mammals,2,34tcwd0j5qgq,84,"41, 10, 76, 68, 38, 9, 77, 70, 59, 43, 26, 17,..."


In [20]:
# Load behavioral data 
df      = pd.read_csv(EXP_DATA)
df_test = df[df["training"] == 0]
print(df_test.shape)
exp_data = df_test.iloc[:,5:]
exp_data.head(3)

(68, 53)


,chp46vl70814,apzvrxiaap22,dp0g9nbjr8wo,ch1i36lf3gb1,mrb0s0ckbda9,bkl2178x4saf,s8hbr6g15kfh,6wstjp4wezt4,9ohtsd1826k8,p4km3aw4hdmt,...,0xhc0yxm5czl,uh7i5dewrmic,wl5925o0j4iv,a3eokjk1z75r,tk04u7pmrsvp,s2edlrqnhaz8,2qewr80masfh,xhkz4vsry78k,rsulb3mqq6p4,u13d9hi208ge
0,2000,1500,1200,700,257,1000,750,600,570,900,...,600,1350,750,2000,600,880,600,700,730,700
1,550,500,1200,600,321,570,550,700,730,900,...,800,802,560,720,400,456,590,700,1100,700
2,3000,2000,2500,360,875,1095,2750,1400,700,1500,...,1500,3520,1566,4000,1500,578,1500,1300,678,570


In [21]:
# For prompt later:
sub_df = df[df["training"] == 1].iloc[:, [1, 4]]

# Build the string
result_string = ", ".join([f"{item} (true value: {crit})" for item, crit in zip(sub_df.iloc[:, 0], sub_df.iloc[:, 1])])

print(result_string)

Red panda (true value: 550), African buffalo (true value: 1475), Wart hog (true value: 578), Tiger (true value: 1268), Bottlenosed dolphin (true value: 2831), Western gray kangaroo (true value: 670), Sugar glider (true value: 236), Stump-tailed macaque (true value: 1186), Slender loris (true value: 380), African bush elephant (true value: 4018), Nutria (true value: 152), Eurasian red squirrel (true value: 296)


In [22]:
# Define cues etc.
cues, ex_cues, ex_crit, ex_ids, test_ids, data = load_data(DOMAIN)

[Mammals] test items: 68, exemplars: 12, participants: 48


In [23]:
# Load negative log-likelihood of Centaur
nll_centaur = torch.stack(torch.load(NLL_PATH, weights_only=True)).float().numpy()
nll_centaur.shape

(48, 68)

In [24]:
AIC_centaur =  2 * 0 + 2 * nll_centaur.sum(-1).sum()
print("AIC_centaur:", AIC_centaur)

AIC_centaur: 26475.48


In [25]:
# load LLM
llm, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen3-32B-bnb-4bit',
    max_seq_length = 32768,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(llm)

==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 2. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

unsloth/Qwen3-32B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 5120, padding_idx=151669)
    (layers): ModuleList(
      (0-63): 64 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=5120, out_features=8192, bias=False)
          (k_proj): Linear4bit(in_features=5120, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=5120, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=8192, out_features=5120, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=5120, out_features=25600, bias=False)
          (up_proj): Linear4bit(in_features=5120, out_features=25600, bias=False)
          (down_proj): Linear4bit(in_features=25600, out_features=5120, bias=False)
          (act_fn): SiLUActivation()
 

In [ ]:
# All participants
n_subs         = exp_data.shape[1]
n_trials       = exp_data.shape[0]
use_model_id   = 0
models         = [model_string_GCM, model_string_CAM]
simulation     = 0
threshold      = 5
best_aic       = float('inf')
last_valid_model_string = models[use_model_id]

while simulation < 10:
    nll            = np.zeros(nll_centaur.shape)
    NUM_PARAMETERS = None
    model          = None
    BOUNDS         = None
    model_string   = models[use_model_id]

    for iteration in range(5):
        try:
            # Model fitting
            exec(model_string)
    
            for participant in range(n_subs):
                # select data
                x = exp_data.iloc[:, participant].values
            
                # model fitting
                x0 = abs(0.01 * np.random.randn(NUM_PARAMETERS))
                
                # Compute NLLs on behavioral data
                res = minimize(nll_truncnorm,
                            x0,
                            args = (model, x, cues, ex_cues, ex_crit, 10000, 'sum'),
                            bounds = BOUNDS,
                            method = 'L-BFGS-B',
                            options={'gtol': 1e-6})
                
                nll[participant] = nll_truncnorm(res.x, model, x, cues, ex_cues, ex_crit, 10000, 'none')
            
            AIC = 2 * NUM_PARAMETERS + 2 * nll.sum(-1)
            current_aic = AIC.sum()
            print(f"Iteration {iteration} - AIC: {current_aic}")
    
            if iteration > 0 and current_aic > (best_aic * 1.5):
                    print(f"Model degraded severely! Rolling back to previous best version.")
                    model_string = last_valid_model_string
                    # Adjust generation parameters slightly to force creative variance
                    temperature_setting = 0.8  
                    continue
            else:
                    best_aic                = current_aic
                    last_valid_model_string = model_string
                    temperature_setting     = 0.6

        except Exception as e:
            print(f"Execution error with LLM code: {e}. Rolling back...")
            model_string = last_valid_model_string
            continue

        # Scientific Regret Minimization
        nll_delta      = nll - nll_centaur
        srm_string     = ""
        num_srm_points = 0

        for participant in range(n_subs):
            # load prompt for participant
            participant_data = dataset.filter(lambda example: example['participant'] == participant)
            n_train_items    = participant_data[0]['n_training']
            
            # print data points
            for trial_id in range(nll_delta.shape[1]):
                substring = until_nth_occurrence(participant_data[0]['text'], "<<", trial_id + n_train_items + 1,95)
                if nll_delta[participant, trial_id].item() > threshold:
                    num_srm_points += 1
                    match    = re.search(r"\[TEST\] Item:\s*(?P<Item>[^.]+)\..*?<<(?P<Estimate>\d+)>>", substring)
                    item     = match.group('Item')
                    estimate = match.group('Estimate')
                    new_string = f"Item {item}. Participant {participant} estimated: {estimate}."
                    srm_string += '* ' + new_string + '\n'
                    
        print("num_srm_points:", num_srm_points)
        
        prompt = (
            "I am studying human behavior in an estimation experiment.\n"
            "In this experiment, participants estimate the number of days until female maturity for various mammals.\n\n"
            "Experiment Structure:\n"
            "1. Training Phase: Participants repeatedly estimated the maturity days for 12 specific exemplar mammals. "
            "After each estimate, they received immediate feedback showing the true value.\n"
            "The 12 learned training exemplars and their true values are: Red panda (true value: 550), African buffalo (true value: 1475), Wart hog (true value: 578),"
            "Tiger (true value: 1268), Bottlenosed dolphin (true value: 2831), Western gray kangaroo (true value: 670), Sugar glider (true value: 236), Stump-tailed macaque (true value: 1186),"
            "Slender loris (true value: 380), African bush elephant (true value: 4018), Nutria (true value: 152), Eurasian red squirrel (true value: 296)\n\n"
            "2. Testing Phase: Participants estimated the maturity days for 80 mammals in a randomized order without any feedback. "
            "This included the 12 old training exemplars and 68 entirely new mammals.\n\n"
            "I have the following computational model that is currently my best guess for how people make estimates in this experiment:\n"
        )
        prompt += model_string
        prompt += '\n\nThis model does capture human behavior reasonably well overall, but there are the following data points in which it does not capture human behavior yet:\n\n'
        prompt += srm_string
        prompt += '\nCan you suggest an improved model that is able to capture human behavior in the listed situations?\n'
        prompt += 'Please structure your answer as follows:\n'
        prompt += '* Keep the structure of the function exactly the same.\n'
        prompt += '* Do not include the sigma parameter anywhere else in the model, only as a second returned object since it is needed for maximum likelihood estimation.\n'
        prompt += '* Crucial: Ensure that EVERY parameter defined in NUM_PARAMETERS is explicitly sliced and used in the code. Do not leave any parameters unused.\n'
        prompt += '* Crucial: If NUM_PARAMETERS is K, ensure your bounds list has exactly K tuples, and your parameters array is sliced accurately from 0 to K-1 (with sigma as the final index parameters[-1]).\n'
        prompt += '* Do not change the docstring.\n'
        prompt += '* State the number of free parameters before the model function using the NUM_PARAMETERS variable.\n'
        prompt += '* State the bounds of the free parameters before the model function using the BOUNDS variable.\n'
        prompt += '* Do not write any text besides that and do not elaborate any further.'
        
        np.savez('../Data/Model Outputs/srm_model_' + str(use_model_id) + '_run_' + str(simulation) + '_iteration_' + str(iteration) + '.npz', nll=nll, num_parameters=NUM_PARAMETERS, prompt=prompt, model_string=model_string)
        
        messages = [
             {"role": "system", "content": ""},
             {"role": "user", "content": prompt},
        ]
        model_outputs = generator(messages, do_sample=True, temperature=temperature_setting, top_p=0.95, top_k=20, min_p=0.1, return_full_text=False, max_new_tokens=16384)
        
        model_string = model_outputs[0]['generated_text'].split("</think>")[-1]
        print(model_string)

Iteration 0 - AIC: 50775.41691713137
num_srm_points: 863


Both `max_new_tokens` (=16384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




NUM_PARAMETERS = 12  
BOUNDS         = [(1e-4, 100)] + [(0, 10)] * 10 + [(1e-3, 10000)]  

def model(parameters, cues, ex_cues, ex_crit, p = 2):  
    """  
    Compute predicted criterion estimate.  

    Parameters  
    ----------  
    parameters : np.ndarray of shape (num_parameters,)  
        Model parameters. sensitivity first, then one weight per cue  

    cues : np.ndarray of shape  (n_trials, n_dim)  
        Feature matrix of stimuli accross trials  

    ex_cues    : np.ndarray of shape (n_exemplars, n_dim)  
        Feature matrix of exemplars  

    ex_crit    : np.ndarray of shape (n_exemplars,)  
        Criterion values for each exemplar  

    Returns  
    -------  
    pred_crit : np.ndarray of shape (n_trials,)  
        Predicted criterion estimates for each trial, clipped to the range [0, 10000]  
    """  

    n_dim     = cues.shape[1]  
    c         = abs(parameters[0])            # sensitivity  
    w         = parameters[1:n_dim + 1]       # feature wei

Both `max_new_tokens` (=16384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


num_srm_points: 810


In [ ]:
# Seperatly
n_subs         = exp_data.shape[1]
n_trials       = exp_data.shape[0]
use_model_id   = 0
models         = [model_string_GCM, model_string_CAM]
threshold      = 5

# Loop through each participant on the top level
for participant in range(n_subs):
    print(f"\n==================================================")
    print(f"OPTIMIZING MODEL FOR PARTICIPANT {participant}")
    print(f"==================================================")
    
    # Reset the model string back to the base model for this participant
    model_string   = models[use_model_id]
    
    # Track the best AIC for just this participant to handle safety rollbacks
    best_aic                = float('inf')
    last_valid_model_string = model_string

    for iteration in range(5):
        try:
            # Model fitting
            exec(model_string)

            # Select data for just this participant
            x = exp_data.iloc[:, participant].values
        
            # Model fitting initialization
            x0 = abs(0.01 * np.random.randn(NUM_PARAMETERS))
            
            # Compute NLLs on behavioral data for this participant
            res = minimize(nll_truncnorm,
                        x0,
                        args = (model, x, cues, ex_cues, ex_crit, 10000, 'sum'),
                        bounds = BOUNDS,
                        method = 'L-BFGS-B',
                        options={'gtol': 1e-6})
            
            # Compute single-participant trial-by-trial NLL array
            p_nll = nll_truncnorm(res.x, model, x, cues, ex_cues, ex_crit, 10000, 'none')
        
            # Compute AIC for this specific participant
            AIC = 2 * NUM_PARAMETERS + 2 * p_nll.sum()
            print(f"Iteration {iteration} - Participant {participant} AIC: {AIC}")

            # Safeguard: Rollback if the LLM generated mathematically broken parameters
            if iteration > 0 and AIC > (best_aic + 50):
                print("Model degraded. Rolling back to previous code layout.")
                model_string = last_valid_model_string
                continue
            else:
                best_aic = AIC
                last_valid_model_string = model_string

        except Exception as e:
            print(f"Execution error with LLM code: {e}. Rolling back...")
            model_string = last_valid_model_string
            continue

        # Scientific Regret Minimization (Comparing participant against their own Centaur baseline)
        nll_delta      = p_nll - nll_centaur[participant]
        srm_string     = ""
        num_srm_points = 0

        # Load prompt for participant
        participant_data = dataset.filter(lambda example: example['participant'] == participant)
        n_train_items    = participant_data[0]['n_training']
        
        # Gather data points where model doesn't match behavior
        for trial_id in range(len(nll_delta)):
            if nll_delta[trial_id].item() > threshold:
                num_srm_points += 1                 
                substring = until_nth_occurrence(participant_data[0]['text'], "<<", trial_id + n_train_items + 1, 95)
                match    = re.search(r"\[TEST\] Item:\s*(?P<Item>[^.]+)\..*?<<(?P<Estimate>\d+)>>", substring)
                item     = match.group('Item')
                estimate = match.group('Estimate')
                new_string = f"Item {item}. You estimated: {estimate}."
                srm_string += '* ' + new_string + '\n'
                    
        print("num_srm_points passed to prompt:", num_srm_points)
      
        prompt = (
            "I am studying human behavior in an estimation experiment.\n"
            "In this experiment, participants estimate the number of days until female maturity for various mammals.\n\n"
            "Experiment Structure:\n"
            "1. Training Phase: Participants repeatedly estimated the maturity days for 12 specific exemplar mammals. "
            "After each estimate, they received immediate feedback showing the true value.\n"
            "The 12 learned training exemplars and their true values are: Red panda (true value: 550), African buffalo (true value: 1475), Wart hog (true value: 578),"
            "Tiger (true value: 1268), Bottlenosed dolphin (true value: 2831), Western gray kangaroo (true value: 670), Sugar glider (true value: 236), Stump-tailed macaque (true value: 1186),"
            "Slender loris (true value: 380), African bush elephant (true value: 4018), Nutria (true value: 152), Eurasian red squirrel (true value: 296)\n\n"
            "2. Testing Phase: Participants estimated the maturity days for 80 mammals in a randomized order without any feedback. "
            "This included the 12 old training exemplars and 68 entirely new mammals.\n\n"
            "I have the following computational model that is currently my best guess for how Participant {participant} made estimates in this experiment:\n"
        )
        prompt += model_string
        prompt += '\n\nThis model does capture human behavior reasonably well overall, but there are the following data points in which it does not capture human behavior yet:\n\n'
        prompt += srm_string
        prompt += '\nCan you suggest an improved model that is able to capture human behavior in the listed situations?\n'
        prompt += 'Please structure your answer as follows:\n'
        prompt += '* Keep the structure of the function exactly the same.\n'
        prompt += '* Do not include the sigma parameter anywhere else in the model, only as a second returned object since it is needed for maximum likelihood estimation.\n'
        prompt += '* Crucial: Ensure that EVERY parameter defined in NUM_PARAMETERS is explicitly sliced and used in the code. Do not leave any parameters unused.\n'
        prompt += '* Crucial: If NUM_PARAMETERS is K, ensure your bounds list has exactly K tuples, and your parameters array is sliced accurately from 0 to K-1 (with sigma as the final index parameters[-1]).\n'
        prompt += '* Do not change the docstring.\n'
        prompt += '* State the number of free parameters before the model function using the NUM_PARAMETERS variable.\n'
        prompt += '* State the bounds of the free parameters before the model function using the BOUNDS variable.\n'
        prompt += '* Do not write any text besides that and do not elaborate any further.'
        
        # Save output state per participant iteration
        np.savez('../Data/Model Outputs/srm_model_' + str(use_model_id) + 'p_' + str(participant) + '_iteration_' + str(iteration) + '.npz', nll=p_nll, num_parameters=NUM_PARAMETERS, prompt=prompt, model_string=model_string)
        
        messages = [
             {"role": "system", "content": ""},
             {"role": "user", "content": prompt},
        ]
        model_outputs = generator(messages, do_sample=True, temperature=0.6, top_p=0.95, top_k=20, min_p=0.1, return_full_text=False, max_new_tokens=16384)
        
        model_string = model_outputs[0]['generated_text'].split("</think>")[-1].strip()